# MAG Quality Control

The first step after recieving the assembled MAGs file is to proceed with the quality control step. Here, we will use [BUSCO](https://busco.ezlab.org/) which uses a set of curated ortholog genes to estimate the quality metrics.

The quality control steps were divided per domain (archaea, bacteria, fungi), as there are no BUSCO databases containing all three, and making it computationally lighter.

**All the following codes were run on Euler. So they cannot be run on Jupyterhub.**

## 1. Archaea
### 1.1 Fetch Archaea Database
We begin by fetching the required archaeal BUSCO database.

In [ ]:
qiime annotate fetch-busco-db \
    --p-lineages archaea_odb12 \
    --o-db $data_dir/busco-db-archaea.qza

### 1.2 Run Busco
We use the fetched database to run BUSCO on our MAGs.

In [ ]:
qiime annotate evaluate-busco \
    --i-mags $data_dir/updog_mags.qza \
    --i-db $data_dir/busco-db-archaea.qza \
    --p-lineage-dataset archaea_odb12 \
    --p-cpu 3 \
    --o-results $data_dir/busco-results-archaea.qza \
    --o-visualization $data_dir/mags-busco-archaea.qzv

### 1.3 Filtering MAGs

We want to keep the MAGs which are at least 50% complete and have less than 10% contamination, which are assessed as medium quality according to the [MIMAG standard](https://doi.org/10.1038/nbt.3893).

In [ ]:
mosh annotate filter-mags \
  --i-mags $data_dir/mags.qza \
  --m-metadata-file $data_dir/busco-results-archaea.qza \
  --p-where "complete > 50 AND contamination < 10" \
  --p-no-exclude-ids \
  --p-on mag \
  --o-filtered-mags $data_dir/mags_filtered_archaea_50.qza \
  --verbose

## 2. Bacteria 
### 2.1 Fetch Bacteria Database
We fetch the required bacterial BUSCO database.

In [ ]:
! qiime annotate fetch-busco-db \
    --p-lineages bacteria_odb12 \
    --o-db $data_dir/busco-db-bacteria.qza

### 2.2 Run Busco
We use the fetched database to run BUSCO on our MAGs.

In [ ]:
! qiime annotate evaluate-busco \
    --i-mags $data_dir/updog_mags.qza \
    --i-db $data_dir/busco-db-bacteria.qza \
    --p-lineage-dataset bacteria_odb12 \
    --p-cpu 3 \
    --o-results $data_dir/busco-results-bacteria.qza \
    --o-visualization $data_dir/mags-busco-bacteria.qzv


### 2.3 MAGs Filtering
Now that we evaluated the quality of our MAGs, we can use this information to filter out only the best ones, with the same parameters as before.

In [ ]:
mosh annotate filter-mags \
  --i-mags $data_dir/mags.qza \
  --m-metadata-file $data_dir/busco-results-bacteria.qza \
  --p-where "complete > 50 AND contamination < 10" \
  --p-no-exclude-ids \
  --p-on mag \
  --o-filtered-mags $data_dir/mags_filtered_bacteria_50.qza \
  --verbose

## 3. Fungi
### 3.1 Fetch Fungi Database
We fetch the required fungal BUSCO database.

In [ ]:
qiime annotate fetch-busco-db \
  --p-lineages fungi_odb12 \
  --o-db $data_dir/busco-db-fungi.qza

### 3.2 Partitioning
As the quality check of fungal MAGs with BUSCO takes too much time (the job times out after five days) and needs more computational capacity in order to run, we partition the MAGs in 100 smaller files

In [ ]:
qiime types partition-sample-data-mags \
  --i-mags updog_mags.qza \
  --p-num-partitions 100 \
  --o-partitioned-mags busco_inputs/updog_mags_partitions

### 3.3 Run Busco

Then we run BUSCO on each individual file:

In [ ]:
QZA_DIR="/cluster/scratch/$USER/updog/busco_inputs/updog_mags_partitions"

# Pick the file corresponding to this task
SAMPLE_FILE=$(ls $QZA_DIR/*.qza | sed -n "${SLURM_ARRAY_TASK_ID}p")

echo "Processing $SAMPLE_FILE on $SLURM_JOB_NODELIST"

# Run BUSCO for fungi
qiime annotate evaluate-busco \
    --i-mags $SAMPLE_FILE \
    --i-db $data_dir/busco-db-fungi.qza \
    --p-lineage-dataset fungi_odb12 \
    --p-cpu 3 \
    --o-results $output_dir/$(basename $SAMPLE_FILE .qza)_busco-results-fungi.qza \
    --o-visualization $output_dir/$(basename $SAMPLE_FILE .qza)_busco-fungi.qzv

### 3.4 MAGs filtering

Then, we filter out the MAGs considered as lower quality using the same parameters on each of the 100 files.

In [ ]:
# Paths
data_dir=/cluster/scratch/$USER/updog
samples_dir=$data_dir/busco_inputs/updog_mags_partitions          # directory with per-sample .qza
busco_metrics=$data_dir/busco-results-bacteria.qza  # your BUSCO results file
output_dir=$data_dir/busco_filtered
mkdir -p $output_dir

# Pick the sample file for this array task
SAMPLE_FILE=$(ls $samples_dir/*.qza | sed -n "${SLURM_ARRAY_TASK_ID}p")
BASENAME=$(basename "$SAMPLE_FILE" .qza)

echo "Filtering $SAMPLE_FILE on $SLURM_JOB_NODELIST"

# Run the filter
mosh annotate filter-mags \
  --i-mags $SAMPLE_FILE \
  --m-metadata-file $busco_metrics \
  --p-where "complete > 50 AND contamination < 10" \
  --p-no-exclude-ids \
  --p-on mag \
  --o-filtered-mags $output_dir/${BASENAME}_filtered.qza \
  --verbose

### 3.5 Collating filtered MAGs

We finally collate all the filtered fungi MAGs in order to only have one file with all of the MAGs for the dereplication step

In [ ]:
qiime types collate-sample-data-mags \
  --i-mags $data_dir/busco_filtered/*.qza \
  --o-collated-mags $data_dir/mags_filtered_all_fungi.qza

We will not merge the QCed files of all three domains just yet, since the dereplication step runs quicker when the files are smaller.